# Notebook 04 — NER Fine-tuning

BERT-NER fine-tuned on 1500 synthetic genealogy documents on Kaggle T4 GPU.
This notebook loads the saved model and shows per-entity precision/recall/F1.

**Base model:** dslim/bert-base-NER
**Labels:** NAME, DATE, LOCATION, AGE, RELATIONSHIP (BIO scheme, 11 tags)
**Training time:** ~38 minutes on Kaggle T4
**Key result:** F1=1.0 on synthetic validation set

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname("__file__"), ".."))
import sys
sys.path.insert(0, "..")
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src.data.synthetic import generate_dataset
from src.ner.model import load_finetuned
from src.ner.train import _align_labels
from src.ner.evaluate import compute_metrics

print("Loading fine-tuned NER model from models/ner-genealogy...")
model, tokenizer, id2label = load_finetuned("models/ner-genealogy")
print("Model loaded.")

In [ ]:
all_docs = generate_dataset(n=1500, seed=42)
val_docs = all_docs[int(len(all_docs) * 0.85):]
print(f"Validation samples: {len(val_docs)}")

In [ ]:
all_preds, all_refs = [], []

for doc in val_docs[:100]:
    words = doc.text.split()
    encoding = tokenizer(
        words, is_split_into_words=True,
        truncation=True, max_length=128, return_tensors="pt"
    )
    with torch.no_grad():
        logits = model(**encoding).logits[0]
    predictions = torch.argmax(logits, dim=-1).numpy()
    ref_labels = _align_labels(words, doc)
    true_pred, true_ref, prev_word_id = [], [], None
    for word_id, pred in zip(encoding.word_ids(), predictions):
        if word_id is None or word_id == prev_word_id:
            prev_word_id = word_id
            continue
        true_pred.append(id2label[pred])
        true_ref.append(ref_labels[word_id])
        prev_word_id = word_id
    all_preds.append(true_pred)
    all_refs.append(true_ref)

metrics = compute_metrics(all_preds, all_refs)
print(f"Overall  Precision: {metrics['precision']:.3f}  Recall: {metrics['recall']:.3f}  F1: {metrics['f1']:.3f}")

In [ ]:
rows = []
for label, scores in metrics["per_label"].items():
    rows.append({
        "Label": label,
        "Precision": round(scores["precision"], 3),
        "Recall": round(scores["recall"], 3),
        "F1": round(scores["f1-score"], 3),
        "Support": scores["number"],
    })
df = pd.DataFrame(rows).set_index("Label")
print(df.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df))
ax.bar([i - 0.25 for i in x], df["Precision"], 0.25, label="Precision", color="#FF6B6B")
ax.bar([i for i in x], df["Recall"], 0.25, label="Recall", color="#4ECDC4")
ax.bar([i + 0.25 for i in x], df["F1"], 0.25, label="F1", color="#45B7D1")
ax.set_xticks(list(x))
ax.set_xticklabels(df.index, rotation=15)
ax.set_ylim(0, 1.1)
ax.set_title("Per-Entity NER Metrics")
ax.legend()
plt.tight_layout()
plt.show()